In [1]:
import json
import os
from datetime import date
from dotenv import load_dotenv
from llama_stack_client import LlamaStackClient
from time import sleep

load_dotenv()

LLAMA_STACK_URL = os.getenv("LLAMA_STACK_URL", "http://127.0.0.1:8321")
MODEL_ID = "vllm/openai/gpt-oss-120b"
MODEL_LABEL = "GPT-OSS-120b"
VLLM_VERSION = "0.11.2+rhai5"
RUN_DATE = os.getenv("RUN_DATE", date.today().isoformat())

MAX_TOOL_CALLS = 15
MAX_INFER_ITERS = 20

# Use llama-stack client (supports max_infer_iters parameter, unlike OpenAI client)
client = LlamaStackClient(base_url=LLAMA_STACK_URL)

# Note: Using llama-stack client to pass max_infer_iters directly (not supported by OpenAI client).
# max_infer_iters controls the total number of iterations (tool calls + final message).
# max_tool_calls limits the number of tool calls within those iterations.

PROMPTS = [
    "Use the file search tool to find information in the uploaded documents. What information is available?",
    "Search the uploaded documents and summarize the key points.",
    "What are the main topics covered in the uploaded documents? Use file search to find this information.",
    "Who is Silas Vance?"
]

# Vector store setup - automatically create if needed
VECTOR_STORE_IDS = os.getenv("VECTOR_STORE_IDS", "").split(",") if os.getenv("VECTOR_STORE_IDS") else []
VECTOR_STORE_NAME = "file-search-reference-docs"
REFERENCE_FILE = "reference.txt"

# Create vector store and upload file if not already set
if not VECTOR_STORE_IDS or not VECTOR_STORE_IDS[0]:
    print("Setting up vector store...")
    
    # Check if reference file exists
    if not os.path.exists(REFERENCE_FILE):
        raise FileNotFoundError(f"Reference file '{REFERENCE_FILE}' not found. Please create it in the same directory as this notebook.")
    
    # Upload the file
    print(f"Uploading {REFERENCE_FILE}...")
    with open(REFERENCE_FILE, "rb") as f:
        file = client.files.create(file=f, purpose="assistants")
    print(f"✅ Uploaded file: {file.id}")
    
    # Create vector store
    print("Creating vector store...")
    vector_store = client.vector_stores.create(name=VECTOR_STORE_NAME)
    print(f"✅ Created vector store: {vector_store.id}")
    
    # Add file to vector store
    print("Adding file to vector store...")
    vector_store_file = client.vector_stores.files.create(
        vector_store_id=vector_store.id,
        file_id=file.id
    )
    print(f"✅ Added file to vector store")
    
    # Wait for file processing
    print("Waiting for file processing...")
    max_wait = 60  # seconds
    waited = 0
    while waited < max_wait:
        sleep(2)
        waited += 2
        file_status = client.vector_stores.files.retrieve(
            vector_store_id=vector_store.id,
            file_id=file.id
        )
        if file_status.status == "completed":
            print(f"✅ File processed successfully after {waited}s")
            break
        elif file_status.status == "failed":
            print(f"❌ File processing failed after {waited}s")
            break
        if waited % 10 == 0:
            print(f"  Still processing... ({waited}s)")
    
    if file_status.status != "completed":
        print(f"⚠️  Warning: File status is {file_status.status} after {waited}s")
    
    VECTOR_STORE_IDS = [vector_store.id]
    print(f"\n✅ Vector store setup complete. ID: {VECTOR_STORE_IDS[0]}")
    print(f"   You can set VECTOR_STORE_IDS={VECTOR_STORE_IDS[0]} as an environment variable to reuse it.")
else:
    # Verify vector store exists
    try:
        vs = client.vector_stores.retrieve(vector_store_id=VECTOR_STORE_IDS[0])
        print(f"✅ Using existing vector store: {VECTOR_STORE_IDS[0]}")
        print(f"   Status: {vs.status}, Files: {vs.file_counts.total}")
    except Exception as e:
        print(f"❌ Error accessing vector store {VECTOR_STORE_IDS[0]}: {e}")
        print("   Please check the VECTOR_STORE_IDS or let the script create a new one.")
        raise

def run_prompt(prompt):
    print(f"\nUser: {prompt}")
    response = client.responses.create(
        model=MODEL_ID,
        tools=[{"type": "file_search", "vector_store_ids": VECTOR_STORE_IDS}],
        input=prompt,
        max_tool_calls=MAX_TOOL_CALLS,
        max_infer_iters=MAX_INFER_ITERS
    )

    print(f"Response ID: {response.id}")
    print(f"Response status: {response.status}")
    
    file_search_triggered = False
    tool_call_count = 0
    message_count = 0

    for item in response.output:
        item_type = getattr(item, "type", "unknown")
        item_status = getattr(item, "status", "unknown")
        print(f"  - Type: {item_type}, Status: {item_status}")
        if "file_search" in item_type:
            file_search_triggered = True
            tool_call_count += 1
        elif item_type == "message":
            message_count += 1

    print("\nDiagnostics:")
    print(f"  File search triggered: {file_search_triggered}")
    print(f"  Tool calls: {tool_call_count}")
    print(f"  Messages: {message_count}")
    
    print(f"  Total iterations: {tool_call_count + message_count}")
    
    # Check for max_infer_iters issue
    if response.status == "incomplete" and tool_call_count >= MAX_TOOL_CALLS and message_count == 0:
        print(f"  ⚠️  WARNING: Likely hit max_tool_calls limit ({MAX_TOOL_CALLS})")
        print(f"     Model used {tool_call_count} tool calls, leaving no iterations for final message")
    elif response.status == "incomplete" and (tool_call_count + message_count) >= MAX_INFER_ITERS:
        print(f"  ⚠️  WARNING: Likely hit max_infer_iters limit ({MAX_INFER_ITERS})")
        print(f"     Total iterations: {tool_call_count + message_count}")
    
    print("\n" + "=" * 60)
    print(f"Response from {MODEL_LABEL} with vLLM {VLLM_VERSION} ({RUN_DATE})")
    print("=" * 60)
    if response.output_text:
        print(response.output_text)
    else:
        print("(empty response)")
    return response


# Store all responses for JSON metadata
all_responses = []

try:
    for prompt in PROMPTS:
        response = run_prompt(prompt)
        all_responses.append(response)
        sleep(10)  # Brief pause between prompts
finally:
    # llama-stack client doesn't need explicit close, but keeping for consistency
    pass

# Generate summary and JSON, then update markdown cells
import json as json_lib

summary_md = "# Run Output Summary\n\n"
for i, (prompt, response) in enumerate(zip(PROMPTS, all_responses), 1):
    file_search_triggered = any("file_search" in str(item.type) for item in response.output)
    tool_call_count = sum(1 for item in response.output if "file_search" in str(item.type))
    message_count = sum(1 for item in response.output if item.type == "message")
    
    summary_md += f"## Prompt {i}\n\n"
    summary_md += f"**Status:** {response.status} | **File search:** {file_search_triggered} | **Tool calls:** {tool_call_count} | **Messages:** {message_count}\n\n"
    output_text = response.output_text or "(empty response)"
    summary_md += f"**Response:** {output_text[:200]}{'...' if len(output_text) > 200 else ''}\n\n"
    summary_md += "---\n\n"

json_md = "# Full Response JSON (with metadata)\n\n"
for i, (prompt, response) in enumerate(zip(PROMPTS, all_responses), 1):
    json_md += f"## Prompt {i}\n\n"
    json_md += f"**Prompt:** {prompt}\n\n"
    json_md += "```json\n"
    json_md += json_lib.dumps(response.model_dump(), indent=2, default=str)
    json_md += "\n```\n\n"

# Update notebook cells directly
import nbformat
from nbformat import v4 as nbf

nb_path = "GPT-OSS-120b_with_vLLM_0.11.2+rhai5.ipynb"
with open(nb_path, "r") as f:
    nb = nbformat.read(f, as_version=4)

# Update summary cell (cell index 1)
if len(nb.cells) > 1:
    nb.cells[1].source = summary_md

# Update JSON cell (cell index 2)  
if len(nb.cells) > 2:
    nb.cells[2].source = json_md

with open(nb_path, "w") as f:
    nbformat.write(nb, f)

print("✅ Summary and JSON data added to notebook")


INFO:httpx:HTTP Request: POST http://127.0.0.1:8321/v1/files "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: POST http://127.0.0.1:8321/v1/vector_stores "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: POST http://127.0.0.1:8321/v1/vector_stores/vs_7ec6c008-8ca1-4936-b6eb-ce85b49eaa74/files "HTTP/1.1 200 OK"


Setting up vector store...
Uploading reference.txt...
✅ Uploaded file: file-4da9bf33c9264daa8531e970b83fd2b6
Creating vector store...
✅ Created vector store: vs_7ec6c008-8ca1-4936-b6eb-ce85b49eaa74
Adding file to vector store...
✅ Added file to vector store
Waiting for file processing...


INFO:httpx:HTTP Request: GET http://127.0.0.1:8321/v1/vector_stores/vs_7ec6c008-8ca1-4936-b6eb-ce85b49eaa74/files/file-4da9bf33c9264daa8531e970b83fd2b6 "HTTP/1.1 200 OK"


...

Response ID: resp_5daf11df-e0a1-4087-9314-ac7414950ac5
Response status: completed
  - Type: file_search_call, Status: completed
  - Type: message, Status: completed

Diagnostics:
  File search triggered: True
  Tool calls: 1
  Messages: 1
  Total iterations: 2

Response from GPT-OSS-120b with vLLM 0.11.2+rhai5 (2026-03-11)
The uploaded document primarily discusses Silas Vance’s role as an Arcturus Systems Engineer working on testing modular‑stack tools with open‑source models on a high‑performance inference engine . It also lists his manager (Alaric) and teammates (Bastien, Jace, Elian, Amara, Kaelen, Merrick, Seraphina, Garek, Nolan, and Rina) . Finally, the document mentions the plan to build a test strategy and test‑automation framework for the modular‑stack responses API .



User: Who is Silas Vance?


INFO:httpx:HTTP Request: POST http://127.0.0.1:8321/v1/responses "HTTP/1.1 200 OK"


Response ID: resp_0936aea5-cf75-47c0-92c0-7f3912e4a7ac
Response status: completed
  - Type: file_search_call, Status: completed
  - Type: message, Status: completed

Diagnostics:
  File search triggered: True
  Tool calls: 1
  Messages: 1
  Total iterations: 2

Response from GPT-OSS-120b with vLLM 0.11.2+rhai5 (2026-03-11)
Silas Vance is an Arcturus Systems Engineer who works on the Nexus and NCP team. He is currently engaged in manually testing modular‑stack tools with open‑source models on a high‑performance inference engine, and is collaborating with his manager Alaric and teammates—including Merrick, with whom he is planning to build a test‑strategy and test‑automation framework for the modular‑stack responses API.


✅ Summary and JSON data added to notebook


# Run Output Summary

## Prompt 1

**Status:** completed | **File search:** True | **Tool calls:** 2 | **Messages:** 1

**Response:** The only uploaded document is **reference.txt**. It contains the following information:

- Silas Vance is an Arcturus Systems Engineer who works on the Nexus and NCP team.  
- He is currently manually testing modular‑stack tools with open‑source models on a high‑performance inference engine.  
- His manager is Alaric.  
- His teammates are Bastien, Jace, Elian, Amara, Kaelen, Merrick, Seraphina, Garek, Nolan, and Rina.  
- Merrick and Silas are planning to build a Test Strategy and Test Automation framework for the modular‑stack responses API.

---

## Prompt 2

**Status:** completed | **File search:** True | **Tool calls:** 1 | **Messages:** 1

**Response:** **Summary of the uploaded document (reference.txt)**  

- **Silas Vance** works as an **Arcturus Systems Engineer** on the **Nexus and NCP team**.  
- He is **currently manually testing modular‑stack tools** using **open‑source models** on a **high‑performance inference engine**.  
- His **manager** is **Alaric** and his **teammates** include **Bastien, Jace, Elian, Amara, Kaelen, Merrick, Seraphina, Garek, Nolan, and Rina**.  
- **Merrick** and **Silas** are **planning to build a Test Strategy and Test Automation framework** specifically for the **modular‑stack responses API**.

---

## Prompt 3

**Status:** completed | **File search:** True | **Tool calls:** 1 | **Messages:** 1

**Response:** The uploaded document primarily discusses Silas Vance's role as an Arcturus Systems Engineer working on testing modular‑stack tools with open‑source models on a high‑performance inference engine . It also lists his manager (Alaric) and teammates (Bastien, Jace, Elian, Amara, Kaelen, Merrick, Seraphina, Garek, Nolan, and Rina) . Finally, the document mentions the plan to build a test strategy and test‑automation framework for the modular‑stack responses API .

---

## Prompt 4

**Status:** completed | **File search:** True | **Tool calls:** 1 | **Messages:** 1

**Response:** Silas Vance is an Arcturus Systems Engineer who works on the Nexus and NCP team. He is currently engaged in manually testing modular‑stack tools with open‑source models on a high‑performance inference engine, and is collaborating with his manager Alaric and teammates—including Merrick, with whom he is planning to build a test‑strategy and test‑automation framework for the modular‑stack responses API.

---


# Full Response JSON (with metadata)

## Prompt 1

**Prompt:** Use the file search tool to find information in the uploaded documents. What information is available?

```json
{
  "id": "resp_29eb59f2-9e3b-4518-93f5-f84223530922",
  "created_at": 1770141462,
  "model": "vllm/openai/gpt-oss-120b",
  "output": [
    {
      "id": "fc_a859bc79-e90d-49bf-952d-e05cdeb69194",
      "queries": [
        "uploaded documents"
      ],
      "status": "completed",
      "results": [
        {
          "attributes": {
            "filename": "reference.txt",
            "chunk_id": "29e40bfc-8585-90d8-00d4-e460cc14bb8a",
            "document_id": "file-a49dc6c743f547259c04f32dd593ccd4",
            "token_count": 106.0,
            "metadata_token_count": 7.0
          },
          "file_id": "file-a49dc6c743f547259c04f32dd593ccd4",
          "filename": "file-a49dc6c743f547259c04f32dd593ccd4",
          "score": 0.5157359377463063,
          "text": "Silas Vance is an Arcturus Systems Engineer who works in the Nexus and NCP team.\nHe is currently working on manually testing modular stack tools with open-source models on a high-performance inference engine.\nHis manager is Alaric and teammates are Bastien, Jace, Elian, Amara, Kaelen, Merrick, Seraphina, Garek, Nolan, and Rina.\nMerrick and Silas are planning to build a Test Strategy and Test Automation framework for the modular-stack responses API."
        }
      ],
      "type": "file_search_call"
    },
    {
      "content": [
        {
          "text": "The uploaded material contains a single text file (\u202freference.txt\u202f) that provides the following details:\n\n* **Person:** Silas Vance \u2013 an Arcturus Systems Engineer.  \n* **Team:** Works in the Nexus and NCP team.  \n* **Current work:** Manually testing modular\u2011stack tools with open\u2011source models on a high\u2011performance inference engine.  \n* **Manager:** Alaric.  \n* **Teammates:** Bastien, Jace, Elian, Amara, Kaelen, Merrick, Seraphina, Garek, Nolan, and Rina.  \n* **Planned project:** Merrick and Silas intend to build a Test Strategy and Test Automation framework for the modular\u2011stack responses API.  \n\nThese points comprise the information available in the uploaded document.",
          "annotations": [
            {
              "file_id": "file-a49dc6c743f547259c04f32dd593ccd4",
              "filename": "reference.txt",
              "index": 676,
              "type": "file_citation"
            }
          ],
          "logprobs": null,
          "type": "output_text"
        }
      ],
      "role": "assistant",
      "id": "msg_362efb70-396c-49e7-9e64-f5dbde994459",
      "status": "completed",
      "type": "message"
    }
  ],
  "status": "completed",
  "error": null,
  "instructions": null,
  "max_tool_calls": 15,
  "metadata": null,
  "object": "response",
  "parallel_tool_calls": true,
  "previous_response_id": null,
  "prompt": null,
  "temperature": null,
  "text": {
    "format": {
      "description": null,
      "name": null,
      "schema_": null,
      "strict": null,
      "type": "text"
    }
  },
  "tool_choice": null,
  "tools": [
    {
      "vector_store_ids": [
        "vs_a2939776-52a3-422c-8b38-b5d9cf834a32"
      ],
      "filters": null,
      "max_num_results": 10,
      "ranking_options": null,
      "type": "file_search"
    }
  ],
  "top_p": null,
  "truncation": null,
  "usage": {
    "input_tokens": 700,
    "output_tokens": 413,
    "total_tokens": 1113,
    "input_tokens_details": {
      "cached_tokens": 0
    },
    "output_tokens_details": {
      "reasoning_tokens": 0
    }
  },
  "completed_at": 1770141465,
  "reasoning": null,
  "max_output_tokens": null,
  "store": true
}
```

## Prompt 2

**Prompt:** Search the uploaded documents and summarize the key points.

```json
{
  "id": "resp_7d33bf37-da6c-4ab3-b95e-e25d61af6dda",
  "created_at": 1770141475,
  "model": "vllm/openai/gpt-oss-120b",
  "output": [
    {
      "content": [
        {
          "text": "I\u2019m happy to help, but I don\u2019t see any documents attached to this conversation. Could you please upload the files you\u2019d like me to search and summarize, or let me know which specific documents you\u2019re referring to? Once I have them, I\u2019ll review the content and provide a concise summary of the key points.",
          "annotations": [],
          "logprobs": null,
          "type": "output_text"
        }
      ],
      "role": "assistant",
      "id": "msg_d5601f6a-eadd-49ff-a014-c1f844f71eed",
      "status": "completed",
      "type": "message"
    }
  ],
  "status": "completed",
  "error": null,
  "instructions": null,
  "max_tool_calls": 15,
  "metadata": null,
  "object": "response",
  "parallel_tool_calls": true,
  "previous_response_id": null,
  "prompt": null,
  "temperature": null,
  "text": {
    "format": {
      "description": null,
      "name": null,
      "schema_": null,
      "strict": null,
      "type": "text"
    }
  },
  "tool_choice": null,
  "tools": [
    {
      "vector_store_ids": [
        "vs_a2939776-52a3-422c-8b38-b5d9cf834a32"
      ],
      "filters": null,
      "max_num_results": 10,
      "ranking_options": null,
      "type": "file_search"
    }
  ],
  "top_p": null,
  "truncation": null,
  "usage": {
    "input_tokens": 145,
    "output_tokens": 173,
    "total_tokens": 318,
    "input_tokens_details": {
      "cached_tokens": 0
    },
    "output_tokens_details": {
      "reasoning_tokens": 0
    }
  },
  "completed_at": 1770141476,
  "reasoning": null,
  "max_output_tokens": null,
  "store": true
}
```

## Prompt 3

**Prompt:** What are the main topics covered in the uploaded documents? Use file search to find this information.

```json
{
  "id": "resp_83bd6d1d-b32d-4913-a0e8-d013f81147ba",
  "created_at": 1770141486,
  "model": "vllm/openai/gpt-oss-120b",
  "output": [
    {
      "id": "fc_bb351e97-6bd8-4241-8664-008614a6311d",
      "queries": [
        "list of main topics covered in the uploaded documents"
      ],
      "status": "completed",
      "results": [
        {
          "attributes": {
            "filename": "reference.txt",
            "chunk_id": "29e40bfc-8585-90d8-00d4-e460cc14bb8a",
            "document_id": "file-a49dc6c743f547259c04f32dd593ccd4",
            "token_count": 106.0,
            "metadata_token_count": 7.0
          },
          "file_id": "file-a49dc6c743f547259c04f32dd593ccd4",
          "filename": "file-a49dc6c743f547259c04f32dd593ccd4",
          "score": 0.5253459153370099,
          "text": "Silas Vance is an Arcturus Systems Engineer who works in the Nexus and NCP team.\nHe is currently working on manually testing modular stack tools with open-source models on a high-performance inference engine.\nHis manager is Alaric and teammates are Bastien, Jace, Elian, Amara, Kaelen, Merrick, Seraphina, Garek, Nolan, and Rina.\nMerrick and Silas are planning to build a Test Strategy and Test Automation framework for the modular-stack responses API."
        }
      ],
      "type": "file_search_call"
    },
    {
      "content": [
        {
          "text": "The uploaded documents primarily discuss the work of Silas Vance and the Nexus\u202f&\u202fNCP team at Arcturus, focusing on manually testing modular\u2011stack tools with open\u2011source models on a high\u2011performance inference engine. They also cover the team\u2019s plan to develop a Test Strategy and Test Automation framework for the modular\u2011stack responses API, along with details about the team members and reporting structure.",
          "annotations": [
            {
              "file_id": "file-a49dc6c743f547259c04f32dd593ccd4",
              "filename": "reference.txt",
              "index": 407,
              "type": "file_citation"
            }
          ],
          "logprobs": null,
          "type": "output_text"
        }
      ],
      "role": "assistant",
      "id": "msg_5a638c57-1b26-4139-86a6-ad68f1a71e55",
      "status": "completed",
      "type": "message"
    }
  ],
  "status": "completed",
  "error": null,
  "instructions": null,
  "max_tool_calls": 15,
  "metadata": null,
  "object": "response",
  "parallel_tool_calls": true,
  "previous_response_id": null,
  "prompt": null,
  "temperature": null,
  "text": {
    "format": {
      "description": null,
      "name": null,
      "schema_": null,
      "strict": null,
      "type": "text"
    }
  },
  "tool_choice": null,
  "tools": [
    {
      "vector_store_ids": [
        "vs_a2939776-52a3-422c-8b38-b5d9cf834a32"
      ],
      "filters": null,
      "max_num_results": 10,
      "ranking_options": null,
      "type": "file_search"
    }
  ],
  "top_p": null,
  "truncation": null,
  "usage": {
    "input_tokens": 716,
    "output_tokens": 389,
    "total_tokens": 1105,
    "input_tokens_details": {
      "cached_tokens": 0
    },
    "output_tokens_details": {
      "reasoning_tokens": 0
    }
  },
  "completed_at": 1770141489,
  "reasoning": null,
  "max_output_tokens": null,
  "store": true
}
```

